# Linear-Probe Evaluation for DINO Checkpoints

Loads a pretrained DINO ViT-Tiny/8 backbone (from `../reducing-data-in-visual-ai/checkpoints/{SPLIT}/dino_e100.pt`), freezes it, attaches a single `nn.Linear(192, 10)` head, trains the head on CIFAR-10 (rescaled to 64x64), and appends a row to `../evaluation/results.csv` with the final test top-1 accuracy.

This is the **paper-faithful linear-probe protocol** per Caron et al. 2021 B.2. The backbone is the teacher (preferred for evaluation per the paper since it's the EMA of the student -> smoother features). Cosine LR, SGD + momentum, no WD on the head.

**To run on all 7 conditions:** flip `CHECKPOINT_SPLIT` in Cell 3 to each of {1024, 2048, 4096, 8192, 16384, 32768, None} where `None` = random-init baseline. Each run appends one row.

In [1]:
import os, sys, pathlib

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    LP_DIR = "/content/drive/MyDrive/research/linear_probing"
    REPO_DIR = "/content/drive/MyDrive/research/reducing-data-in-visual-ai"
    EVAL_DIR = "/content/drive/MyDrive/research/evaluation"
    %cd $LP_DIR
    !pip install -q datasets
else:
    LP_DIR = str(pathlib.Path().resolve())
    REPO_DIR = str(pathlib.Path(LP_DIR).parent / "reducing-data-in-visual-ai")
    EVAL_DIR = str(pathlib.Path(LP_DIR).parent / "evaluation")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"LP_DIR   = {LP_DIR}")
print(f"REPO_DIR = {REPO_DIR}")
print(f"EVAL_DIR = {EVAL_DIR}")

Mounted at /content/drive
/content/drive/MyDrive/research/linear_probing
LP_DIR   = /content/drive/MyDrive/research/linear_probing
REPO_DIR = /content/drive/MyDrive/research/reducing-data-in-visual-ai
EVAL_DIR = /content/drive/MyDrive/research/evaluation


## Cell 2 - Imports

In [2]:
import json, math, time, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# Repo imports (sys.path was set in Cell 1)
from models.dino import ViTBackbone, DINOHead
from shared_config import VIT_TINY_8_KWARGS

# Local import (cifar10_utils.py is in the same folder)
from cifar10_utils import get_cifar10_dataloaders

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

PyTorch: 2.10.0+cu128, CUDA: True


## Cell 3 - Hyperparameters & which checkpoint to evaluate

In [3]:
# Which DINO pretraining fraction to evaluate. Set to None for the random-init baseline.
CHECKPOINT_SPLIT = 1024     # one of: 1024, 2048, 4096, 8192, 16384, 32768, None

# Linear-probe training hyperparameters (DINO Caron 2021 App. B.2 / MoCo v3 main_lincls.py protocol).
# SGD with linear LR scaling: LR = 0.1 * batch_size / 256.
EVAL_EPOCHS  = 100
EVAL_BATCH   = 128
EVAL_LR_BASE = 0.1          # base SGD LR; actual LR = EVAL_LR_BASE * EVAL_BATCH / 256 (set in Cell 7 after batch override)
EVAL_MOMENTUM = 0.9
EVAL_WD      = 0.0          # paper: no WD on linear-probe head
RESCALE      = 64           # ViT-Tiny/8 input size; matches DINO pretrain
SEED         = 0

# Backbone source: teacher (paper preferred) or student.
USE_TEACHER  = True

# Derive checkpoint path.
ckpt_path = None if CHECKPOINT_SPLIT is None else \
    pathlib.Path(REPO_DIR) / "checkpoints" / str(CHECKPOINT_SPLIT) / "dino_e100.pt"

if CHECKPOINT_SPLIT is None:
    METHOD_NAME = "random_init"
    FRACTION = 0
    print("RANDOM-INIT BASELINE - no checkpoint will be loaded")
else:
    METHOD_NAME = "dino"
    FRACTION = CHECKPOINT_SPLIT
    print(f"Will load: {ckpt_path}")
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"


Will load: /content/drive/MyDrive/research/reducing-data-in-visual-ai/checkpoints/1024/dino_e100.pt


## Cell 4 - Seed & device (TF32 enabled on Ampere+)

In [4]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Hardware-adaptive tuning: A100 (Colab) vs laptop GPU.
# Overrides EVAL_BATCH and NUM_WORKERS from Cell 3 when A100 is detected.
if device.type == "cuda" and "A100" in torch.cuda.get_device_name(0):
    EVAL_BATCH = 512
    NUM_WORKERS = 8
    AMP_DTYPE = torch.bfloat16
    USE_AMP = True
    print(f"A100 detected -> EVAL_BATCH={EVAL_BATCH}, NUM_WORKERS={NUM_WORKERS}, AMP=bf16 (no scaler)")
else:
    AMP_DTYPE = torch.float16
    USE_AMP = False
    print(f"Non-A100 GPU -> EVAL_BATCH={EVAL_BATCH}, NUM_WORKERS={NUM_WORKERS}, AMP=disabled")

# Apply linear LR scaling now that EVAL_BATCH is final.
EVAL_LR = EVAL_LR_BASE * EVAL_BATCH / 256
print(f"EVAL_LR = {EVAL_LR:.4f}  (= EVAL_LR_BASE={EVAL_LR_BASE} * EVAL_BATCH={EVAL_BATCH} / 256)")


Device: cuda
GPU: NVIDIA A100-SXM4-40GB
A100 detected -> EVAL_BATCH=512, NUM_WORKERS=8, AMP=bf16 (no scaler)
EVAL_LR = 0.2000  (= EVAL_LR_BASE=0.1 * EVAL_BATCH=512 / 256)


## Cell 5 - Load CIFAR-10 at 64x64

In [5]:
# Auto-tune num_workers for the device. 0 on Windows-without-frozen-helpers, otherwise 4.
# NUM_WORKERS already set by Cell 4

train_loader, test_loader = get_cifar10_dataloaders(
    seed=SEED, rescale=RESCALE, batch=EVAL_BATCH, num_workers=NUM_WORKERS,
)
print(f"Train batches: {len(train_loader)}  ({len(train_loader.dataset)} samples)")
print(f"Test  batches: {len(test_loader)}  ({len(test_loader.dataset)} samples)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Train batches: 97  (50000 samples)
Test  batches: 20  (10000 samples)


## Cell 6 - Build backbone (load DINO teacher, or fresh random init)

In [6]:
# Build the same nn.Sequential the dino_pretrain notebook used.
full_model = nn.Sequential(ViTBackbone(), DINOHead()).to(device)

if ckpt_path is not None:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_key = "teacher" if USE_TEACHER else "student"
    full_model.load_state_dict(ckpt[state_key], strict=True)
    print(f"Loaded {state_key} from {ckpt_path.name}")
else:
    print("Skipping checkpoint load - random-init baseline")

# We only need the BACKBONE (the ViT) for linear probing. Discard the DINOHead.
backbone = full_model[0]    # ViTBackbone - outputs the [CLS] token, shape [B, 192]
del full_model              # free the DINOHead from memory

# Freeze the backbone.
backbone.eval()
for p in backbone.parameters():
    p.requires_grad_(False)

# Verify output shape with one dummy batch.
with torch.no_grad():
    dummy = torch.zeros(2, 3, RESCALE, RESCALE, device=device)
    feat = backbone(dummy)
print(f"Backbone output shape: {feat.shape}  (expect [2, 192])")
assert feat.shape == (2, 192), f"Unexpected feature shape: {feat.shape}"

Loaded teacher from dino_e100.pt
Backbone output shape: torch.Size([2, 192])  (expect [2, 192])


## Cell 7 - Linear classifier head

In [7]:
classifier = nn.Linear(192, 10).to(device)
print(f"Classifier params: {sum(p.numel() for p in classifier.parameters()):,}")

Classifier params: 1,930


## Cell 8 - Optimizer + cosine LR schedule (head only)

In [8]:
optim = torch.optim.SGD(
    classifier.parameters(),
    lr=EVAL_LR, momentum=EVAL_MOMENTUM, weight_decay=EVAL_WD,
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EVAL_EPOCHS, eta_min=0)
print(f"Optimizer: SGD lr={EVAL_LR} momentum={EVAL_MOMENTUM} wd={EVAL_WD}")
print(f"Schedule:  Cosine over {EVAL_EPOCHS} epochs -> 0")

Optimizer: SGD lr=0.2 momentum=0.9 wd=0.0
Schedule:  Cosine over 100 epochs -> 0


## Cell 9 - Training loop with per-epoch test eval

In [9]:
# def evaluate(backbone, classifier, loader, device):
#     """Returns test top-1 accuracy (%)."""
#     classifier.eval()
#     correct, total = 0, 0
#     with torch.no_grad():
#         for imgs, labels in loader:
#             imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
#             with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
#                 feat = backbone(imgs)
#             if USE_AMP:
#                 feat = feat.float()
#             logits = classifier(feat)
#             preds = logits.argmax(dim=-1)
#             correct += (preds == labels).sum().item()
#             total += labels.size(0)
#     return 100.0 * correct / total


# t0 = time.time()
# best_test_acc = 0.0
# history = []

# for epoch in range(EVAL_EPOCHS):
#     classifier.train()
#     running_loss = 0.0
#     running_correct = 0
#     running_total = 0
#     pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EVAL_EPOCHS}", unit="step")
#     for imgs, labels in pbar:
#         imgs   = imgs.to(device, non_blocking=True)
#         labels = labels.to(device, non_blocking=True)

#         with torch.no_grad():
#             with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
#                 feat = backbone(imgs)        # frozen - no grads
#             if USE_AMP:
#                 feat = feat.float()

#         logits = classifier(feat)
#         loss = F.cross_entropy(logits, labels)

#         optim.zero_grad()
#         loss.backward()
#         optim.step()

#         running_loss += loss.item() * labels.size(0)
#         running_correct += (logits.argmax(-1) == labels).sum().item()
#         running_total += labels.size(0)
#         pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{sched.get_last_lr()[0]:.2e}")

#     sched.step()

#     train_loss = running_loss / running_total
#     train_acc  = 100.0 * running_correct / running_total
#     test_acc   = evaluate(backbone, classifier, test_loader, device)
#     best_test_acc = max(best_test_acc, test_acc)

#     history.append({"epoch": epoch+1, "train_loss": train_loss,
#                     "train_acc": train_acc, "test_acc": test_acc})
#     print(f"Epoch {epoch+1}/{EVAL_EPOCHS}: train_loss={train_loss:.4f} train_acc={train_acc:.2f}% test_acc={test_acc:.2f}%")

# wallclock = time.time() - t0
# final_test_acc = history[-1]["test_acc"]

# print()
# print(f"=== {METHOD_NAME} fraction={FRACTION} ===")
# print(f"Best test top-1 accuracy: {best_test_acc:.2f}%")
# print(f"Final test top-1 accuracy: {final_test_acc:.2f}%")
# print(f"Wallclock: {wallclock:.1f}s")

## Cell 10 - Save linear-head checkpoint + append CSV row

In [10]:
# # Save the trained linear head + history. ONLY in linear_probing/, never touches reducing-data-in-visual-ai/.
# ckpt_split_dir = pathlib.Path(LP_DIR) / "linear_probe_checkpoints" / str(FRACTION)
# ckpt_split_dir.mkdir(parents=True, exist_ok=True)

# head_path = ckpt_split_dir / f"head_e{EVAL_EPOCHS}_s{SEED}.pt"
# torch.save(
#     {
#         "classifier": classifier.state_dict(),
#         "config": {
#             "CHECKPOINT_SPLIT": CHECKPOINT_SPLIT,
#             "EVAL_EPOCHS": EVAL_EPOCHS, "EVAL_BATCH": EVAL_BATCH,
#             "EVAL_LR": EVAL_LR, "EVAL_MOMENTUM": EVAL_MOMENTUM, "EVAL_WD": EVAL_WD,
#             "RESCALE": RESCALE, "SEED": SEED, "USE_TEACHER": USE_TEACHER,
#         },
#         "history": history,
#         "best_test_acc": best_test_acc,
#         "final_test_acc": final_test_acc,
#         "wallclock": wallclock,
#     },
#     head_path,
# )
# print(f"Head saved -> {head_path}  ({head_path.stat().st_size / 1024:.1f} KB)")


# # Append row to evaluation/results.csv. Create with header if file doesn't exist.
# import csv
# from datetime import datetime
# results_path = pathlib.Path(EVAL_DIR) / "results.csv"
# results_path.parent.mkdir(parents=True, exist_ok=True)

# # Group-shared schema from Next Steps Week 4 CSV schemas.
# header = [
#     "method", "fraction", "seed", "epochs",
#     "eval_dataset", "eval_method",
#     "accuracy", "checkpoint_path",
#     "wallclock_pretrain_s", "wallclock_eval_s",
#     "git_sha", "timestamp",
# ]

# row = {
#     "method": METHOD_NAME,
#     "fraction": FRACTION,
#     "seed": SEED,
#     "epochs": EVAL_EPOCHS,
#     "eval_dataset": "cifar10",
#     "eval_method": "linear_probe",
#     "accuracy": f"{best_test_acc:.4f}",
#     "checkpoint_path": "" if ckpt_path is None else str(ckpt_path.relative_to(pathlib.Path(LP_DIR).parent)),
#     "wallclock_pretrain_s": "",     # unknown here; filled in later if desired
#     "wallclock_eval_s": f"{wallclock:.1f}",
#     "git_sha": "",                  # filled in later if running from a clean repo
#     "timestamp": datetime.now().isoformat(timespec="seconds"),
# }

# file_existed = results_path.exists()
# with open(results_path, "a", newline="", encoding="utf-8") as f:
#     writer = csv.DictWriter(f, fieldnames=header)
#     if not file_existed:
#         writer.writeheader()
#     writer.writerow(row)
# print(f"Appended row to {results_path}")
# print(f"  method={row['method']}  fraction={row['fraction']}  accuracy={row['accuracy']}")

## Cell 11 — Batch Mode: Run all curve points in one execution

**Skip cells 11-19 above; use this cell instead** to evaluate all 7 conditions sequentially in one execution.

For each `split` in `[1024, 2048, 4096, 8192, 16384, 32768, None]`:
- Loads the pretrained DINO teacher backbone from `../reducing-data-in-visual-ai/checkpoints/{split}/dino_e100.pt` (random init if `split is None`)
- Trains a fresh `nn.Linear(192, 10)` head on CIFAR-10 for `EVAL_EPOCHS` epochs
- Saves the head to `linear_probe_checkpoints/{FRACTION}/head_e{EVAL_EPOCHS}_s{SEED}.pt`
- Appends one row to `../evaluation/results.csv`

The driver re-seeds at the start of each split (so each is reproducible) and `del`s + `torch.cuda.empty_cache()` between iterations to avoid VRAM creep.

**Expected wallclock on A100:** ~90 min total (~13 min per condition × 7).
**Expected wallclock on laptop 3060:** ~3-4 hr total — better to run on A100.

**Use cells 11-19 above** for single-checkpoint debugging or to inspect a specific run's training curve; use this cell for the full data-efficiency sweep.

In [ ]:
# Batch driver — runs the full linear-probe sweep across all curve points.
# Each iteration rebuilds backbone, classifier, optimizer, and scheduler from
# scratch and re-seeds the RNGs so each split is independently reproducible.

SPLITS_TO_EVAL = [None, 1024, 2048, 4096, 8192, 16384, 32768]

import csv
from datetime import datetime


def _evaluate_test(backbone, classifier, loader, device):
    """Test-set top-1 accuracy (%). Self-contained so this cell doesn't depend on Cell 9."""
    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                feat = backbone(imgs)
            if USE_AMP:
                feat = feat.float()
            logits = classifier(feat)
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


def _run_one(checkpoint_split):
    """Run a single linear-probe eval. Returns (best_test_acc, wallclock_s)."""
    method_name = "random_init" if checkpoint_split is None else "dino"
    fraction = 0 if checkpoint_split is None else checkpoint_split

    # Re-seed for per-iteration reproducibility
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    print()
    print("=" * 60)
    print(f"  Evaluating: method={method_name}, fraction={fraction}")
    print("=" * 60)

    # --- Build backbone -------------------------------------------------
    full_model = nn.Sequential(ViTBackbone(), DINOHead()).to(device)
    if checkpoint_split is not None:
        ckpt_path = pathlib.Path(REPO_DIR) / "checkpoints" / str(checkpoint_split) / "dino_e100.pt"
        assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        state_key = "teacher" if USE_TEACHER else "student"
        full_model.load_state_dict(ckpt[state_key], strict=True)
        print(f"Loaded {state_key} from {ckpt_path.name}")
    else:
        print("Random-init baseline -- no checkpoint loaded")

    backbone = full_model[0]
    del full_model
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad_(False)

    # --- Build classifier + optimizer + scheduler ----------------------
    classifier = nn.Linear(192, 10).to(device)
    optim = torch.optim.SGD(
        classifier.parameters(),
        lr=EVAL_LR, momentum=EVAL_MOMENTUM, weight_decay=EVAL_WD,
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EVAL_EPOCHS, eta_min=0)

    # --- Train ----------------------------------------------------------
    t0 = time.time()
    best_test_acc = 0.0
    history = []

    for epoch in range(EVAL_EPOCHS):
        classifier.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.no_grad():
                with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                    feat = backbone(imgs)
                if USE_AMP:
                    feat = feat.float()

            logits = classifier(feat)
            loss = F.cross_entropy(logits, labels)

            optim.zero_grad()
            loss.backward()
            optim.step()

            running_loss += loss.item() * labels.size(0)
            running_correct += (logits.argmax(-1) == labels).sum().item()
            running_total += labels.size(0)

        sched.step()

        train_loss = running_loss / running_total
        train_acc = 100.0 * running_correct / running_total
        test_acc = _evaluate_test(backbone, classifier, test_loader, device)
        best_test_acc = max(best_test_acc, test_acc)
        history.append({"epoch": epoch + 1, "train_loss": train_loss,
                        "train_acc": train_acc, "test_acc": test_acc})

        # Print only every 10th epoch (and first/last) to keep logs short
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == EVAL_EPOCHS - 1:
            print(f"  Epoch {epoch+1}/{EVAL_EPOCHS}: "
                  f"train_loss={train_loss:.4f} train_acc={train_acc:.2f}% test_acc={test_acc:.2f}%")

    wallclock = time.time() - t0
    final_test_acc = history[-1]["test_acc"]

    print()
    print(f"  Best test top-1:  {best_test_acc:.2f}%")
    print(f"  Final test top-1: {final_test_acc:.2f}%")
    print(f"  Wallclock:        {wallclock:.1f}s")

    # --- Save linear-head checkpoint -----------------------------------
    ckpt_split_dir = pathlib.Path(LP_DIR) / "linear_probe_checkpoints" / str(fraction)
    ckpt_split_dir.mkdir(parents=True, exist_ok=True)
    head_path = ckpt_split_dir / f"head_e{EVAL_EPOCHS}_s{SEED}.pt"
    torch.save(
        {
            "classifier": classifier.state_dict(),
            "config": {
                "CHECKPOINT_SPLIT": checkpoint_split,
                "EVAL_EPOCHS": EVAL_EPOCHS, "EVAL_BATCH": EVAL_BATCH,
                "EVAL_LR": EVAL_LR, "EVAL_MOMENTUM": EVAL_MOMENTUM, "EVAL_WD": EVAL_WD,
                "RESCALE": RESCALE, "SEED": SEED, "USE_TEACHER": USE_TEACHER,
                "USE_AMP": USE_AMP, "AMP_DTYPE": str(AMP_DTYPE),
            },
            "history": history,
            "best_test_acc": best_test_acc,
            "final_test_acc": final_test_acc,
            "wallclock": wallclock,
        },
        head_path,
    )
    print(f"  Head saved -> {head_path}")

    # --- Append CSV row -------------------------------------------------
    results_path = pathlib.Path(EVAL_DIR) / "results.csv"
    results_path.parent.mkdir(parents=True, exist_ok=True)
    header = [
        "method", "fraction", "seed", "epochs",
        "eval_dataset", "eval_method",
        "accuracy", "checkpoint_path",
        "wallclock_pretrain_s", "wallclock_eval_s",
        "git_sha", "timestamp",
    ]
    if checkpoint_split is None:
        ckpt_relpath = ""
    else:
        ckpt_rel = (pathlib.Path(REPO_DIR) / "checkpoints" / str(checkpoint_split) / "dino_e100.pt")
        ckpt_relpath = str(ckpt_rel.relative_to(pathlib.Path(LP_DIR).parent))
    row = {
        "method": method_name,
        "fraction": fraction,
        "seed": SEED,
        "epochs": EVAL_EPOCHS,
        "eval_dataset": "cifar10",
        "eval_method": "linear_probe",
        "accuracy": f"{best_test_acc:.4f}",
        "checkpoint_path": ckpt_relpath,
        "wallclock_pretrain_s": "",
        "wallclock_eval_s": f"{wallclock:.1f}",
        "git_sha": "",
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    file_existed = results_path.exists() and results_path.stat().st_size > 0
    with open(results_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        if not file_existed:
            writer.writeheader()
        writer.writerow(row)
    print(f"  Appended row to {results_path.name}")

    # --- Memory cleanup before next iteration --------------------------
    del backbone, classifier, optim, sched
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_test_acc, wallclock


# Run the sweep
t_sweep = time.time()
results_summary = []
for split in SPLITS_TO_EVAL:
    acc, wc = _run_one(split)
    results_summary.append((split, acc, wc))

# Final summary table
total = time.time() - t_sweep
print()
print("=" * 60)
print(f"  SWEEP COMPLETE -- {total:.1f}s total ({total/60:.1f} min)")
print("=" * 60)
print(f"{'condition':>14} {'accuracy':>10} {'wallclock':>12}")
print("-" * 40)
for split, acc, wc in results_summary:
    name = "random_init" if split is None else f"dino-{split}"
    print(f"{name:>14} {acc:>9.2f}% {wc:>10.1f}s")


  Evaluating: method=random_init, fraction=0
Random-init baseline -- no checkpoint loaded
  Epoch 1/100: train_loss=2.1852 train_acc=24.87% test_acc=28.24%
  Epoch 10/100: train_loss=1.9784 train_acc=31.12% test_acc=30.44%
  Epoch 20/100: train_loss=1.8735 train_acc=33.71% test_acc=33.86%
  Epoch 30/100: train_loss=1.8611 train_acc=34.06% test_acc=35.29%
  Epoch 40/100: train_loss=1.7933 train_acc=35.68% test_acc=35.71%
  Epoch 50/100: train_loss=1.7431 train_acc=37.19% test_acc=37.84%
  Epoch 60/100: train_loss=1.7143 train_acc=38.48% test_acc=38.99%
  Epoch 70/100: train_loss=1.6887 train_acc=39.34% test_acc=39.27%
  Epoch 80/100: train_loss=1.6774 train_acc=39.88% test_acc=39.72%
  Epoch 90/100: train_loss=1.6628 train_acc=40.68% test_acc=39.91%
  Epoch 100/100: train_loss=1.6568 train_acc=40.80% test_acc=40.30%

  Best test top-1:  40.49%
  Final test top-1: 40.30%
  Wallclock:        766.7s
  Head saved -> /content/drive/MyDrive/research/linear_probing/linear_probe_checkpoints/0/